# 02. Phase 1 Encoder Baselines

Репродукционный notebook для baseline-экспериментов Phase 1:
`RuBERT/XLM-R` на `text_ru/text_masked`.

In [ ]:
%pip install -q torch transformers scikit-learn sentencepiece accelerate evaluate

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "research_baselines").exists():
    pass

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from research_baselines.training import BaselineTrainConfig, GenericBaselineTrainer
from research_baselines.aggregation import aggregate_run_directories

DATA_RAW = PROJECT_ROOT / "data" / "cocolofa_ru_v2.jsonl"
DATA_MASKED = PROJECT_ROOT / "data" / "cocolofa_ru_v2_masked.jsonl"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "phase1_baselines"

RUN_TRIAL = False
RUN_FULL_GRID = False
SEEDS = [42, 52, 62]

BASELINE_MATRIX = [
    {"run_name": "rubert_text_ru", "dataset_path": DATA_RAW, "model_name": "DeepPavlov/rubert-base-cased", "text_column": "text_ru"},
    {"run_name": "rubert_text_masked", "dataset_path": DATA_MASKED, "model_name": "DeepPavlov/rubert-base-cased", "text_column": "text_masked"},
    {"run_name": "xlmr_text_ru", "dataset_path": DATA_RAW, "model_name": "FacebookAI/xlm-roberta-base", "text_column": "text_ru"},
    {"run_name": "xlmr_text_masked", "dataset_path": DATA_MASKED, "model_name": "FacebookAI/xlm-roberta-base", "text_column": "text_masked"},
]

planned_runs = []
for entry in BASELINE_MATRIX:
    for seed in SEEDS:
        planned_runs.append({**entry, "seed": seed})

if RUN_TRIAL:
    planned_runs = planned_runs[:1]

pd.DataFrame(planned_runs)

In [ ]:
completed_runs = []

if RUN_TRIAL or RUN_FULL_GRID:
    for run in planned_runs:
        config = BaselineTrainConfig(
            dataset_path=str(run["dataset_path"]),
            model_name=run["model_name"],
            text_column=run["text_column"],
            seed=int(run["seed"]),
            output_dir=str(OUTPUT_ROOT / run["run_name"] / f"seed_{run['seed']}"),
            num_epochs=5,
            batch_size=16,
            eval_batch_size=32,
            learning_rate=2e-5,
            max_length=512,
            patience=2,
        )
        trainer = GenericBaselineTrainer(config)
        result = trainer.run()
        completed_runs.append(
            {
                "run_name": run["run_name"],
                "seed": run["seed"],
                "text_column": run["text_column"],
                "macro_f1": result["metrics_test"]["macro_f1"],
            }
        )

pd.DataFrame(completed_runs)

In [ ]:
if OUTPUT_ROOT.exists():
    summary = aggregate_run_directories(OUTPUT_ROOT, PROJECT_ROOT / "artifacts" / "summary")
    summary
else:
    print("No run directories found yet. Run training first or unpack full results archive.")